In [1]:
pip install langchain-community

In [2]:
import os
from langchain_community.document_loaders import TextLoader
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline
from langchain_text_splitters import MarkdownHeaderTextSplitter

/tmp/ipykernel_21924/3161468112.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [4]:
from langchain_community.document_loaders import TextLoader
loader=TextLoader("/content/tennis.md")
text_doc=loader.load()
print(text_doc)

[Document(metadata={'source': '/content/tennis.md'}, page_content="# Tennis\n## Introduction\nTennis is a popular sport played between two players (singles) or two teams of two players each (doubles). The game involves using a racket to hit a ball over a net into the opponent's court.\n##Basic Rules\n- A match can be played as best of three or five sets.\n- Each set consists of games, and each game consists of points.\n- Points are scored as 0 (Love), 15, 30, 40, and then game.\n- A player must win a game by at least two points.\n- The ball must land within the designated court boundaries.\n## Scoring System\n0 points  -> Love\n1 point   -> 15\n2 points  -> 30\n3 points  -> 40\n4 points  -> Game (if leading by 2)\n\nDeuce     -> 40-40 (must win two consecutive points to win the game)\nAdvantage -> If a player wins a point at deuce, they gain the advantage\n\n\n## Famous Tournaments\n- Grand Slam Events:\n- Australian Open\n- French Open\n- Wimbledon\n- US Open\n## Equipment\n- Racket: 

In [5]:
# from langchain_text_splitters import MarkdownHeaderTextSplitter
# split_condition=[("##","title")]
# splitter = MarkdownHeaderTextSplitter(split_condition)
# doc_split = splitter.split_text(text_doc[0].page_content)
# print(doc_split)

from langchain_text_splitters import MarkdownHeaderTextSplitter

split_condition = [("##", "title")]
splitter = MarkdownHeaderTextSplitter(split_condition)

doc_splits = splitter.split_text(text_doc[0].page_content)
doc_list = [split.page_content for split in doc_splits]

print(doc_list)

['# Tennis', "Tennis is a popular sport played between two players (singles) or two teams of two players each (doubles). The game involves using a racket to hit a ball over a net into the opponent's court.\n##Basic Rules\n- A match can be played as best of three or five sets.\n- Each set consists of games, and each game consists of points.\n- Points are scored as 0 (Love), 15, 30, 40, and then game.\n- A player must win a game by at least two points.\n- The ball must land within the designated court boundaries.", '0 points  -> Love\n1 point   -> 15\n2 points  -> 30\n3 points  -> 40\n4 points  -> Game (if leading by 2)  \nDeuce     -> 40-40 (must win two consecutive points to win the game)\nAdvantage -> If a player wins a point at deuce, they gain the advantage', '- Grand Slam Events:\n- Australian Open\n- French Open\n- Wimbledon\n- US Open', '- Racket: Used to hit the ball.']


In [42]:
len(doc_splits)

5

In [6]:
embedding_model=SentenceTransformer("all-MiniLM-L6-v2")
def embedded_chunk(chunk):
  return embedding_model.encode([chunk])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
sample_embedding=embedded_chunk("doc_list[0]").tolist()[0]
print(sample_embedding)

[-0.0356583409011364, 0.12018264830112457, 0.028967710211873055, 0.0006921487511135638, 0.06281089782714844, -0.006324221380054951, 0.04787154495716095, 0.09744758903980255, -0.015054202638566494, 0.026374399662017822, 0.012131660245358944, 0.05120643973350525, 0.04059532657265663, -0.0022955697495490313, -0.1009424552321434, 0.0034668361768126488, -0.10517456382513046, -0.005734720267355442, 0.06844403594732285, 0.024177169427275658, 0.020335817709565163, 0.19529372453689575, -0.07301371544599533, 0.03049573488533497, -0.007464537862688303, 0.013653869740664959, -0.11065243929624557, -0.06144791096448898, -0.0652565211057663, -0.0083686001598835, 0.040539320558309555, 0.012836173176765442, 0.05884462594985962, -0.016243355348706245, 0.10934094339609146, 0.011575235053896904, 0.08622099459171295, -0.06146467849612236, -0.020239222794771194, 0.10203158855438232, 0.003201600629836321, -0.06870462000370026, -0.09970307350158691, -0.010569910518825054, -0.03367755562067032, 0.0086270309984

In [8]:
pip install chromadb

In [12]:
descriptions = Chroma.from_texts(
    doc_list,
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"),
    persist_directory="/tmp/chroma_db"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [15]:
descriptions._collection.get(include=["embeddings","documents"])

{'ids': ['fb7dd7dd-5ce5-4f2a-9e09-21220fa9dfb0',
  'ddb5ee3c-eb9c-410f-b3e7-fef98b71f59a',
  '77192bd0-927d-40b7-bd67-8d232d354892',
  '108cb459-9dfb-4116-8470-85881115620b',
  '0d25d469-c4c6-4eed-b3d8-76e2973d0680'],
 'embeddings': array([[ 0.02275678,  0.05737349,  0.06708644, ..., -0.091282  ,
          0.03132664,  0.02229538],
        [ 0.05575813,  0.00622989,  0.03024485, ..., -0.00349529,
         -0.02546689,  0.03141049],
        [-0.03361608,  0.03077638, -0.03815167, ...,  0.03109943,
          0.01729568, -0.01060182],
        [ 0.01498185, -0.00239695,  0.0187146 , ..., -0.09686654,
         -0.0458901 ,  0.03546912],
        [ 0.06801429,  0.134855  , -0.0150764 , ..., -0.04488423,
          0.01263765,  0.03549522]], shape=(5, 384)),
 'documents': ['# Tennis',
  "Tennis is a popular sport played between two players (singles) or two teams of two players each (doubles). The game involves using a racket to hit a ball over a net into the opponent's court.\n##Basic Rules\n- 

### 1. Simple Similarity Search
You can directly query your vector database to find the most semantically similar chunks to a given question.

In [16]:
query = "What are the four Grand Slam tournaments?"
results = descriptions.similarity_search(query, k=2)

for i, doc in enumerate(results):
    print(f"--- Match {i+1} ---")
    print(doc.page_content)
    print()

--- Match 1 ---
- Grand Slam Events:
- Australian Open
- French Open
- Wimbledon
- US Open

--- Match 2 ---
# Tennis



---

---

---

### 4. Using Free, Local Open Models (No API Key Required)
We can load an open-source model like `google/flan-t5-base` directly into the Colab notebook memory using the Hugging Face `transformers` library. This runs entirely on your allocated Colab resources and does not require any keys.

In [24]:
from transformers import pipeline
import torch

# Initialize a lightweight instruction-following text-generation model (SmolLM2-135M)
print("Downloading and initializing the local model...")
local_generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-135M-Instruct",
    device=0 if torch.cuda.is_available() else -1
)
print("Local model is ready!")

config.json:   0%|          | 0.00/861 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

Local model is ready!


In [25]:
def ask_local_model_with_db(question):
    # 1. Retrieve the most relevant document chunks from Chroma
    relevant_docs = descriptions.similarity_search(question, k=2)
    context = "\n".join([doc.page_content for doc in relevant_docs])

    # 2. Use a chat template format to instruct the model
    messages = [
        {
            "role": "system",
            "content": "Answer the question based ONLY on the provided context. Be extremely concise."
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}"
        }
    ]

    # 3. Format the prompt and generate
    prompt = local_generator.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    response = local_generator(prompt, max_new_tokens=100, temperature=0.1, do_sample=True)

    # 4. Extract generated answer
    generated_text = response[0]['generated_text']
    answer = generated_text[len(prompt):].strip()
    return answer

# Test our completely free and local RAG pipeline!
try:
    user_question = "What are the four Grand Slam tournaments?"
    answer = ask_local_model_with_db(user_question)
    print(f"Question: {user_question}\n")
    print(f"Local AI Answer:\n{answer}")
except Exception as e:
    print(f"Error executing local RAG: {e}")

Question: What are the four Grand Slam tournaments?

Local AI Answer:
The four Grand Slam tournaments are:

1. Grand Slam Events:
- Grand Slam: The most prestigious and prestigious tennis tournament in the world.
- Australian Open: The most prestigious and prestigious tennis tournament in the world.
- French Open: The most prestigious and prestigious tennis tournament in the world.
- Wimbledon: The most prestigious and prestigious tennis tournament in the world.

Answer: Grand Slam Events.
